In [0]:
import sys
from pathlib import Path

current = Path.cwd()

repo_root = next(
    (
        path
        for path in [current, *current.parents]
        if (
            path / "src" / "house_price_simple_analysis"
        ).is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        f"Project root not found from {current}"
    )

src_path = str(repo_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from house_price_simple_analysis.config import ProjectConfig

config = ProjectConfig.from_yaml(
    str(repo_root / "config" / "project_config.yml")
)

## Initialize the class

In [0]:
from house_price_simple_analysis.model_release import ModelRelease

release = ModelRelease(
    config=config,
    endpoint_name="house-price-endpoint"
)

## inspect the current deployment

In [0]:
champion = release.lifecycle.get_model_by_alias("Champion")
deployed_version = release.get_deployed_version()

print("Champion version:", champion.version)
print("Deployed version:", deployed_version)


## Test the promotion safeguard

In [0]:
candidate = release.lifecycle.get_model_by_alias("Candidate")

release.promote_and_deploy(
    version=candidate.version,
    apply=False
)

Since that Candidate was registered before we introduced validation evidence, the method should raise a ValueError.

That's expected: the release process is rejecting a model that hasn't passed our version-specific validation requirements.

No endpoint or alias changes should occur.

run the next code just if you have a new valid candidate.

In [0]:
release.promote_and_deploy(
    version=candidate.version,
    apply=True
)

In [0]:
prediction = release.deployment.predict(
    square_footage=1000,
    num_bedrooms=3,
    num_bathrooms=2,
    year_built=2015,
    lot_size=3500,
    garage_size=2,
    neighborhood_quality=7
)

print(prediction.predictions)

### Considerations for experimenting with multiple models
- **Table schemas**: When adding or removing columns, your Delta tables may need schema updates. In your environment, this requires explicit schema management using sql.



- **Model compatibility**: Your validator can compare models with different feature sets as long as the evaluation dataset contains all the features required by both models. Be careful not to remove columns needed by the current Champion.



- **Serving requests**: Each deployed model still requires its own expected inputs. Your flexible predict(**features) method handles this without code changes, but the caller must supply the correct features.
